# 03 — Modelos, verossimilhança e rasters (`models`, `raster_io`)

Origem: script 11.

## O modelo

`fit_model` devolve um dicionário **autocontido**:

| Chave | Conteúdo |
|---|---|
| `distribution` | nome no catálogo |
| `ENL` | ENL usado (ou `None`) |
| `features` | atributos do treino, na ordem |
| `classes` | ordem das classes (= colunas da verossimilhança) |
| `params` | `{classe: parâmetros}` |

Como distribuição, ENL e atributos ficam gravados no modelo, a aplicação usa obrigatoriamente o que foi usado na estimação. Atributos são localizados **pelo nome**: colunas fora de ordem ou extras são aceitas, e atributos ausentes geram erro.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))   # pasta que contém sar_classification

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sar_classification import (distributions as dist, samples as smp, models as mdl,
                                raster_io as rio, separability as sep, validation as val,
                                fitting as fit, plots, diagnostics as diag)

In [2]:
# Dados fictícios gerados pelo próprio catálogo (substitua pelos seus df_samples)
rng = np.random.default_rng(42)
classes = ['F', 'VS', 'PA', 'SE']

def simulate(distribution, params_by_class, n, columns, ENL=None):
    parts = [pd.DataFrame(dist.get_distribution(distribution)['sample'](n, p, ENL, rng), columns=columns).assign(Class=c)
             for c, p in params_by_class.items()]
    return pd.concat(parts, ignore_index=True)[['Class'] + columns]

def random_cov(scale):
    A = rng.normal(0, scale, (3, 3))
    return A @ A.T + 1e-4 * np.eye(3)

df_gauss = simulate('gaussian', {c: {'mean': np.array(m), 'covariance_matrix': random_cov(0.02)} for c, m in
                    {'F': [0.20, 0.05, 0.30], 'VS': [0.22, 0.07, 0.28], 'PA': [0.10, 0.15, 0.20], 'SE': [0.05, 0.25, 0.10]}.items()},
                    120, ['B1', 'B2', 'B3'])
df_gamma = simulate('gamma', {c: {'alpha': 3.0, 'beta': 3.0 / mu} for c, mu in
                    {'F': 0.08, 'VS': 0.06, 'PA': 0.03, 'SE': 0.015}.items()}, 150, ['HH'], ENL=3)
df_pair = simulate('intensity_joint_distribution', {c: {'mu1': a, 'mu2': b, 'ro2': r} for c, (a, b, r) in
                   {'F': (0.08, 0.020, 0.25), 'VS': (0.06, 0.012, 0.16), 'PA': (0.03, 0.006, 0.09), 'SE': (0.015, 0.004, 0.49)}.items()},
                   150, ['HH', 'HV'], ENL=4)
print(df_gauss.shape, df_gamma.shape, df_pair.shape)

(480, 4) (600, 2) (600, 3)


## 1. Estimação e verossimilhança

| Função | Descrição |
|---|---|
| `fit_model(df, classes, distribution='gaussian', ENL=None, method=None)` | parâmetros por classe (`method=None` → padrão do catálogo) |
| `class_likelihood(data, model, log=False)` | `DataFrame` n × classes |
| `maximum_likelihood_classifier(likelihoods)` | nome da classe de maior valor (Eq. 2.5; empate → 1ª coluna) |
| `classify(data, model)` | atalho: log-verossimilhança + MaxVer |
| `save_model(model, path)`, `load_model(path)` | JSON |

**Use `log=True` para classificar.** O arg max é o mesmo, sem risco de *underflow*.

In [3]:
train, valid = smp.split_samples(df_pair, ratio=0.7, random_state=1)
model = mdl.fit_model(train, classes, 'intensity_joint_distribution', ENL=4)
pd.DataFrame(model['params']).T

,mu1,mu2,ro2
F,0.079162,0.020233,0.176120
VS,0.064631,0.012973,0.248995
PA,0.028391,0.006957,0.020085
SE,0.015360,0.003901,0.265088


In [4]:
loglik = mdl.class_likelihood(valid, model, log=True)   # 'Class' é ignorada; HH e HV são localizadas pelo nome
pred = mdl.maximum_likelihood_classifier(loglik)
print('Acerto bruto:', round((pred.to_numpy() == valid['Class'].to_numpy()).mean(), 4))
loglik.head()

Acerto bruto: 0.65


,F,VS,PA,SE
0,5.673350,6.075411,2.103828,-5.652642
2,6.189188,6.615752,5.210487,0.835370
3,4.850559,4.330108,3.264541,-4.142959
6,3.346196,1.933791,0.459640,-10.965827
11,5.358542,4.887547,-3.208963,-13.180130


In [5]:
import tempfile
path = Path(tempfile.mkdtemp()) / 'modelo_par.json'
mdl.save_model(model, path)
same = np.allclose(mdl.class_likelihood(valid, mdl.load_model(path), log=True), loglik)
print(f'Modelo salvo em {path.name}; recarregado reproduz a verossimilhança: {same}')

Modelo salvo em modelo_par.json; recarregado reproduz a verossimilhança: True


In [6]:
# Erro explícito quando os dados não correspondem ao modelo
try:
    mdl.class_likelihood(valid[['HH']], model)
except ValueError as e:
    print('ValueError:', e)

ValueError: Atributos do modelo ausentes nos dados: ['HV'] (modelo: ['HH', 'HV']).


## 2. Rasters

| Função | Descrição |
|---|---|
| `raster_to_table(path, band_names=None)` | `(tabela pixels × bandas, meta)`; ordem de linha; nodata → NaN |
| `table_to_raster(table, meta, output_path, dtype, nodata)` | GeoTIFF; descrição de cada banda = nome da coluna |
| `likelihood_raster(image_path, model, output_path, band_names=None, log=False)` | imagem → verossimilhança por classe |

Em `likelihood_raster`:

- `band_names=None` assume que as bandas da imagem são os atributos do modelo, na mesma ordem.
- Pixels com nodata em qualquer banda saem como NaN.
- `log=False` grava verossimilhança, o formato do `cmap_classifier` atual. Valores abaixo de ~10⁻³⁸ viram 0 em `float32`; nesse caso, a função avisa quantos pixels foram afetados.

In [7]:
import rasterio
from rasterio.transform import from_origin
work = Path(tempfile.mkdtemp())
H, W = 30, 40
img = np.stack([rng.gamma(4, 0.02, (H, W)), rng.gamma(4, 0.005, (H, W))]).astype('float32')
with rasterio.open(work / 'par.tif', 'w', driver='GTiff', height=H, width=W, count=2, dtype='float32',
                   crs='EPSG:31983', transform=from_origin(0, 0, 25, 25)) as dst:
    dst.write(img)

out = rio.likelihood_raster(work / 'par.tif', model, work / 'lik_par.tif')
with rasterio.open(out) as src:
    print('Bandas:', src.descriptions, '| dimensões:', src.shape)

Bandas: ('F', 'VS', 'PA', 'SE') | dimensões: (30, 40)
